# Persiapan Dependencies dan GPU

In [1]:
!pip install -q transformers==4.56.2
!pip install -q bitsandbytes==0.49.2
!pip install -q accelerate==1.13.0
!pip install -q peft==0.18.1
!pip install -q "datasets==4.3.0"

# Unsloth 
import torch, re, os
v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
!pip install sentencepiece protobuf "huggingface_hub>=0.34.0" hf_transfer -q
!pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth -q
!pip install --no-deps trl==0.22.2 -q
!pip install torchao>=0.16.0 -q

# Library khusus RAG 
!pip install -q langchain
!pip install -q langchain-community
!pip install -q langchain-core
!pip install -q langchain-text-splitters
!pip install -q langchain-huggingface
!pip install -q chromadb
!pip install -q sentence-transformers
!pip install -q rank-bm25
!pip install -q pdfplumber
!pip install -q pymupdf
!pip install ddgs -q
!pip install -q gradio


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 86.5 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 103.3 MB/s eta 0:00:0000:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba

In [2]:
import subprocess

libraries = [
    "langchain", "langchain-community", "faiss-cpu",
    "sentence-transformers", "rank-bm25", "pypdf",
    "pdfplumber", "gradio", "ddgs",
    "unsloth", "transformers", "cross-encoder",
]

print("VERSI LIBRARY")
for lib in libraries:
    try:
        result = subprocess.run(["pip", "show", lib], capture_output=True, text=True)
        for line in result.stdout.split("\n"):
            if line.startswith("Version:"):
                version = line.split(": ")[1].strip()
                print(f"{lib:<25} == {version}")
    except Exception:
        print(f"{lib:<25} -- tidak ditemukan")

VERSI LIBRARY
langchain                 == 1.2.15
langchain-community       == 0.4.2
sentence-transformers     == 5.4.0
rank-bm25                 == 0.2.2
pypdf                     == 6.12.1
pdfplumber                == 0.11.10
gradio                    == 5.50.0
ddgs                      == 9.14.4
unsloth                   == 2026.6.9
transformers              == 4.56.2


In [3]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    pipeline
)
from langchain_huggingface import HuggingFacePipeline, HuggingFaceEmbeddings
from langchain_community.document_loaders import PDFPlumberLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from sentence_transformers import CrossEncoder
from ddgs import DDGS
from kaggle_secrets import UserSecretsClient
from IPython.display import Markdown, display
import torch
import os
import re
import gradio as gr

2026-06-25 13:53:14.918284: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782395595.161525      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782395595.231602      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782395595.791714      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782395595.791755      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782395595.791758      58 computation_placer.cc:177] computation placer alr

In [4]:
class EnsembleRetriever:
    """
    Menggabungkan BM25 (keyword) dan Chroma (semantic)
    dengan Reciprocal Rank Fusion berbobot
    """
    def __init__(self, retrievers, weights):
        self.retrievers = retrievers
        self.weights    = weights

    def invoke(self, query, k=5):
        all_results = []
        for retriever, weight in zip(self.retrievers, self.weights):
            try:
                docs = retriever.invoke(query)
            except AttributeError:
                docs = retriever.get_relevant_documents(query)

            for rank, doc in enumerate(docs):
                all_results.append({
                    "doc"   : doc,
                    "rank"  : rank,
                    "weight": weight
                })

        # Reciprocal Rank Fusion berbobot
        doc_scores = {}
        doc_map    = {}
        for result in all_results:
            content   = result["doc"].page_content
            rrf_score = result["weight"] / (result["rank"] + 60)
            if content not in doc_scores:
                doc_scores[content] = 0.0
                doc_map[content]    = result["doc"]
            doc_scores[content] += rrf_score

        sorted_docs = sorted(
            doc_scores.keys(),
            key     = lambda x: doc_scores[x],
            reverse = True
        )
        return [doc_map[c] for c in sorted_docs[:k]]

In [5]:
class CrossEncoderReranker:
    """
    Rerank dokumen menggunakan Cross-Encoder
    dan ekstrak relevance score Top-1
    """
    def __init__(self, model_name, top_n=3):
        print(f"Memuat reranker: {model_name}")
        self.model  = CrossEncoder(model_name)
        self.top_n  = top_n
        print(f"Reranker siap! Top-{top_n} dokumen akan diambil")

    def rerank(self, query, documents):
        """Return (top_docs, top1_score)"""
        if not documents:
            return [], 0.0

        pairs  = [(query, doc.page_content) for doc in documents]
        scores = self.model.predict(pairs)

        scored_docs = sorted(
            zip(scores, documents),
            key     = lambda x: x[0],
            reverse = True
        )

        top_docs   = [doc for _, doc in scored_docs[:self.top_n]]
        top1_score = float(scored_docs[0][0]) if scored_docs else 0.0

        return top_docs, top1_score

reranker = CrossEncoderReranker(
    model_name = "cross-encoder/ms-marco-MiniLM-L-6-v2",
    top_n      = 3
)

print("\nSemua komponen manual siap!")
print("  - EnsembleRetriever (RRF berbobot)")
print("  - CrossEncoderReranker (Top-3)")

Memuat reranker: cross-encoder/ms-marco-MiniLM-L-6-v2


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Reranker siap! Top-3 dokumen akan diambil

Semua komponen manual siap!
  - EnsembleRetriever (RRF berbobot)
  - CrossEncoderReranker (Top-3)


# Login Huggingface

In [6]:
secrets  = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
from huggingface_hub import login
login(token=hf_token)
print("Login HF berhasil!")

if torch.cuda.is_available():
    gpu_stats  = torch.cuda.get_device_properties(0)
    max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
    print(f"GPU: {gpu_stats.name} | VRAM: {max_memory} GB")
else:
    print("WARNING: GPU tidak terdeteksi!")

Login HF berhasil!
GPU: Tesla T4 | VRAM: 14.562 GB


# Load Model GRPO

In [7]:
MODEL_NAME = "Rahmat15/qwen2.5-3b-indonesian-legal-grpo"

print(f"Memuat model: {MODEL_NAME}")

# BitsAndBytesConfig untuk 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_use_double_quant = True,    
    bnb_4bit_quant_type       = "nf4",
    bnb_4bit_compute_dtype    = torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token = hf_token
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config = bnb_config,
    device_map          = "auto",
    token               = hf_token
)

# Buat pipeline (ikuti pola latihan)
text_generation_pipeline = pipeline(
    model              = model,
    tokenizer          = tokenizer,
    task               = "text-generation",
    temperature        = 0.3,
    do_sample          = True,
    repetition_penalty = 1.1,
    return_full_text   = False,
    max_new_tokens     = 512,
)

llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

print("Model berhasil dimuat!")

Memuat model: Rahmat15/qwen2.5-3b-indonesian-legal-grpo


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/240 [00:00<?, ?B/s]

Device set to use cuda:0


Model berhasil dimuat!


# Load PDF & Metadata Enrichment

In [8]:
# Sesuaikan PDF_DIR dan nama file dengan output cell di atas
PDF_DIR = "/kaggle/input/datasets/rahmatramadhanrama/legal-documents/"  

# Metadata enrichment untuk setiap UU
PDF_FILES = [
    {
        "path"   : f"{PDF_DIR}PP Nomor 35 Tahun 2021.pdf",
        "nama_uu": "PP No. 35 Tahun 2021",
        "topik"  : "PKWT, Alih Daya, Waktu Kerja dan waktu istirahat, dan pemutusah hubungan kerja",
        "sumber" : "Pemerintah RI"
    },
    {
        "path"   : f"{PDF_DIR}PP Nomor 5 Tahun 2021.pdf",
        "nama_uu": "PP No. 5 Tahun 2021",
        "topik"  : "Penyelenggaraan Perizinan Berusaha Berbasis Risiko",
        "sumber" : "Pemerintah RI"
    },
    {
        "path"   : f"{PDF_DIR}PP Nomor 51 Tahun 2023.pdf",
        "nama_uu": "PP No. 51 Tahun 2023",
        "topik"  : "Pengupahan",
        "sumber" : "Pemerintah RI"
    },
    {
        "path"   : f"{PDF_DIR}UU Nomor 6 Tahun 2023.pdf",
        "nama_uu": "UU No. 6 Tahun 2023",
        "topik"  : "Penetapan Perppu Cipta Kerja",
        "sumber" : "DPR RI"
    },
]

# Load semua PDF dengan metadata enrichment
print("Memuat dokumen PDF...")
all_documents = []

for pdf_info in PDF_FILES:
    if not os.path.exists(pdf_info["path"]):
        print(f"File tidak ditemukan: {pdf_info['path']}")
        continue

    loader = PDFPlumberLoader(pdf_info["path"])
    pages  = loader.load()

    for i, page in enumerate(pages):
        page.metadata.update({
            "nama_uu"   : pdf_info["nama_uu"],
            "topik"     : pdf_info["topik"],
            "sumber"    : pdf_info["sumber"],
            "page"      : i + 1,
            "total_page": len(pages),
        })
        all_documents.append(page)

print(f"Total halaman dimuat: {len(all_documents)}")
print(f"\nContoh metadata:")
print(all_documents[0].metadata)

Memuat dokumen PDF...
Total halaman dimuat: 1949

Contoh metadata:
{'source': '/kaggle/input/datasets/rahmatramadhanrama/legal-documents/PP Nomor 35 Tahun 2021.pdf', 'file_path': '/kaggle/input/datasets/rahmatramadhanrama/legal-documents/PP Nomor 35 Tahun 2021.pdf', 'page': 1, 'total_pages': 56, 'CreationDate': "D:20210218155405+07'00'", 'ModDate': "D:20210218160705+07'00'", 'Creator': 'Canon ', 'Producer': ' ', 'nama_uu': 'PP No. 35 Tahun 2021', 'topik': 'PKWT, Alih Daya, Waktu Kerja dan waktu istirahat, dan pemutusah hubungan kerja', 'sumber': 'Pemerintah RI', 'total_page': 56}


# Chunking Parent-Child

In [9]:
# Child chunks: kecil untuk pencarian
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size    = 512,  
    chunk_overlap = 64,   
    separators    = ["\n\n", "\n", ".", " "]
)

# Parent chunks: besar untuk konteks LLM
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size    = 2048,
    chunk_overlap = 128,
    separators    = ["\n\n", "\n", ".", " "]
)

child_chunks  = child_splitter.split_documents(all_documents)
parent_chunks = parent_splitter.split_documents(all_documents)

print(f"Child chunks  : {len(child_chunks)}")
print(f"Parent chunks : {len(parent_chunks)}")
print(f"\nContoh child chunk:")
print(f"Content : {child_chunks[0].page_content[:200]}")
print(f"Metadata: {child_chunks[0].metadata}")

Child chunks  : 5261
Parent chunks : 1961

Contoh child chunk:
Content : SALINAN
PRESIDEN
REPUBLIK INDONESIA
PERATURAN PEMERINTAH REPUBLIK INDONESIA
NOMOR 35 TAHUN 2O2I
TENTANG
PERJANJIAN KERJA WAKTU TERTENTU, ALIH DAYA, WAKTU KERJA DAN
WAKTU ISTIRAHAT, DAN PEMUTUSAN HUBUN
Metadata: {'source': '/kaggle/input/datasets/rahmatramadhanrama/legal-documents/PP Nomor 35 Tahun 2021.pdf', 'file_path': '/kaggle/input/datasets/rahmatramadhanrama/legal-documents/PP Nomor 35 Tahun 2021.pdf', 'page': 1, 'total_pages': 56, 'CreationDate': "D:20210218155405+07'00'", 'ModDate': "D:20210218160705+07'00'", 'Creator': 'Canon ', 'Producer': ' ', 'nama_uu': 'PP No. 35 Tahun 2021', 'topik': 'PKWT, Alih Daya, Waktu Kerja dan waktu istirahat, dan pemutusah hubungan kerja', 'sumber': 'Pemerintah RI', 'total_page': 56}


# Embedding & Chroma Vector Store

In [10]:
EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

print(f"Memuat embedding: {EMBEDDING_MODEL}")
embeddings = HuggingFaceEmbeddings(
    model_name   = EMBEDDING_MODEL,
    model_kwargs = {"device": "cuda"},
    encode_kwargs= {"normalize_embeddings": True}
)
print("Embedding model siap!")

# Buat Chroma vector store dari child chunks
chroma_vectorstore = Chroma.from_documents(
    documents         = child_chunks,
    embedding         = embeddings,
    persist_directory = "/kaggle/working/chroma_db"
)

print(f"\nChroma vector store siap!")
print(f" Total chunks: {chroma_vectorstore._collection.count()}")

Memuat embedding: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model siap!

Chroma vector store siap!
 Total chunks: 5261


# Ensemble Retriever

In [11]:
# BM25 Retriever (keyword-based)
bm25_retriever   = BM25Retriever.from_documents(child_chunks)
bm25_retriever.k = 5

# Chroma Retriever (semantic-based)
chroma_retriever = chroma_vectorstore.as_retriever(
    search_kwargs = {"k": 5}
)

# Ensemble Retriever dengan bobot (class manual)
ensemble_retriever = EnsembleRetriever(
    retrievers = [bm25_retriever, chroma_retriever],
    weights    = [0.4, 0.6]
)

print("Ensemble Retriever siap!")
print(" BM25   weight: 0.4 (keyword)")
print(" Chroma weight: 0.6 (semantic)")

# Test retrieve minimal 5 dokumen
test_docs = ensemble_retriever.invoke("hak lembur karyawan", k=5)
print(f"\nTest retrieval: {len(test_docs)} dokumen ditemukan")
print(f"\nContoh dokumen pertama:")
print(f"Content : {test_docs[0].page_content[:150]}")
print(f"Metadata: nama_uu={test_docs[0].metadata.get('nama_uu')} | page={test_docs[0].metadata.get('page')}")

Ensemble Retriever siap!
 BM25   weight: 0.4 (keyword)
 Chroma weight: 0.6 (semantic)

Test retrieval: 5 dokumen ditemukan

Contoh dokumen pertama:
Content : PRESiDEN
REPUBLIK INDONESI,A.
-18-
(21 Perintah dan persetujuan sebagaimana dimaksud
pada ayat (1) dapat dibuat dalam bentuk daftar
Pekerja/Buruh yang
Metadata: nama_uu=PP No. 35 Tahun 2021 | page=18


# Reranker

In [12]:
# CrossEncoder Reranker
class CrossEncoderReranker:
    def __init__(self, model_name, top_n=3):
        print(f"Memuat reranker: {model_name}")
        self.model = CrossEncoder(model_name)
        self.top_n = top_n
        print(f"Reranker siap! Top-{top_n} dokumen akan diambil")

    def rerank(self, query, documents):
        """Return (top_docs, top1_score)"""
        if not documents:
            return [], 0.0

        pairs  = [(query, doc.page_content) for doc in documents]
        scores = self.model.predict(pairs)

        scored_docs = sorted(
            zip(scores, documents),
            key     = lambda x: x[0],
            reverse = True
        )

        top_docs   = [doc for _, doc in scored_docs[:self.top_n]]
        top1_score = float(scored_docs[0][0]) if scored_docs else 0.0

        print(f"Relevance Score Top-1: {top1_score:.4f}")
        return top_docs, top1_score

# Inisialisasi reranker
reranker = CrossEncoderReranker(
    model_name = "cross-encoder/ms-marco-MiniLM-L-6-v2",
    top_n      = 3
)

# Test reranker
print("\nTest reranker...")
test_reranked, test_score = reranker.rerank(
    "hak lembur karyawan",
    test_docs
)
print(f"Dokumen setelah rerank : {len(test_reranked)}")
print(f"Top-1 relevance score  : {test_score:.4f}")
print(f"Top-1 nama_uu          : {test_reranked[0].metadata.get('nama_uu')}")

Memuat reranker: cross-encoder/ms-marco-MiniLM-L-6-v2
Reranker siap! Top-3 dokumen akan diambil

Test reranker...
Relevance Score Top-1: -2.4761
Dokumen setelah rerank : 3
Top-1 relevance score  : -2.4761
Top-1 nama_uu          : PP No. 35 Tahun 2021


# HyDE

In [13]:
def generate_hypothetical_answers(query, n=2):
    """
    HyDE: Generate minimal 2 jawaban hipotetis
    untuk memperkaya query sebelum retrieval
    """
    hypothetical_answers = []

    for i in range(n):
        hyde_prompt = f"""Jawab pertanyaan berikut berdasarkan pengetahuan 
hukum ketenagakerjaan Indonesia secara singkat dan spesifik:

Pertanyaan: {query}
Jawaban singkat:"""

        result = text_generation_pipeline(
            hyde_prompt,
            max_new_tokens = 150,
            temperature    = 0.8,
            do_sample      = True,
            return_full_text = False,
        )
        answer = result[0]["generated_text"].strip()
        hypothetical_answers.append(answer)
        print(f"Jawaban hipotetis {i+1}: {answer[:150]}...")

    return hypothetical_answers

# Test HyDE
print("Test HyDE...")
test_hyde = generate_hypothetical_answers("hak lembur karyawan", n=2)
print(f"\nHyDE menghasilkan {len(test_hyde)} jawaban hipotetis")

Test HyDE...
Jawaban hipotetis 1: Dalam hukum ketenagakerjaan Indonesia, hak lembur diberikan kepada karyawan yang terdaftar di bawah sistem pemberlakukan lembur sesuai dengan aturan y...
Jawaban hipotetis 2: Hukum Ketentuan Pajak dan Tenaga Kerja Indonesia menetapkan bahwa karyawan dapat mengambil lembur dengan persetujuan manajemen perusahaan, sepanjang w...

HyDE menghasilkan 2 jawaban hipotetis


# DuckDuckGo Fallback

In [14]:
RELEVANCE_THRESHOLD = -3.0

def duckduckgo_search(query, max_results=3):
    """Fallback ke internet jika relevance score < threshold"""
    print(f"Mencari di internet: {query}")
    results = []
    try:
        ddgs = DDGS()
        search_results = list(ddgs.text(
            f"{query} hukum ketenagakerjaan Indonesia",
            max_results = max_results
        ))

        for r in search_results:
            results.append(Document(
                page_content = r.get("body", ""),
                metadata     = {
                    "source" : r.get("href", ""),
                    "title"  : r.get("title", ""),
                    "type"   : "internet_search",
                    "nama_uu": "Internet Search",
                    "topik"  : "Online",
                    "page"   : "-"
                }
            ))
        print(f"{len(results)} hasil dari internet")

    except Exception as e:
        print(f"Search gagal: {e}")

    return results

# Test
print("Test DuckDuckGo Search...")
test_ddg = duckduckgo_search("hak lembur karyawan", max_results=3)
print(f"Hasil: {len(test_ddg)} dokumen")
if test_ddg:
    print(f"Contoh: {test_ddg[0].page_content[:150]}")

Test DuckDuckGo Search...
Mencari di internet: hak lembur karyawan
3 hasil dari internet
Hasil: 3 dokumen
Contoh: May 3, 2026 · Pahami aturan lembur karyawan di Indonesia lengkap dengan dasar hukum, syarat sah lembur, dan cara menghitung upah lembur sesuai regulas


# Prompt Template & RAG Chain:

In [15]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

template = """Anda adalah asisten hukum AI berbahasa Indonesia.
Anda WAJIB menjawab dengan format berikut TANPA PENGECUALIAN:

<think>
[Tuliskan analisis dan proses berpikir Anda berdasarkan konteks hukum yang diberikan]
</think>
[Tuliskan jawaban final Anda di sini dalam bahasa Indonesia]

Contoh format yang benar:
User: Apakah karyawan kontrak berhak mendapat THR?
Assistant: <think>
Karyawan kontrak (PKWT) diatur dalam UU Ketenagakerjaan No. 13 Tahun 2003.
Berdasarkan Permenaker No. 6 Tahun 2016, karyawan PKWT yang telah bekerja
minimal 1 bulan berhak mendapat THR secara proporsional sesuai masa kerja.
</think>
Ya, karyawan kontrak berhak mendapat THR secara proporsional berdasarkan
Permenaker No. 6 Tahun 2016.

Gunakan HANYA informasi dari konteks berikut untuk menjawab pertanyaan.

Konteks:
{context}

Pertanyaan: {question}
Jawaban:"""

prompt = PromptTemplate(
    template        = template,
    input_variables = ["context", "question"]
)

def format_context(docs):
    return "\n\n".join([doc.page_content for doc in docs])

print("Prompt template siap!")

Prompt template siap!


# Pipeline RAG Lengkap

In [16]:
def rag_pipeline(question):
    """Pipeline RAG lengkap: HyDE → Ensemble → Rerank → Fallback → Generate"""
    print(f"\n{'='*60}")
    print(f"Pertanyaan: {question}")
    print(f"{'='*60}")

    # STEP 1: HyDE
    print("\nStep 1: HyDE - Generating hypothetical answers...")
    hypothetical_answers = generate_hypothetical_answers(question, n=2)

    # Gabungkan query asli + jawaban hipotetis untuk retrieval
    hyde_queries  = [question] + hypothetical_answers
    all_retrieved = []
    seen_contents = set()

    for q in hyde_queries:
        docs = ensemble_retriever.invoke(q, k=5)
        for doc in docs:
            if doc.page_content not in seen_contents:
                all_retrieved.append(doc)
                seen_contents.add(doc.page_content)

    print(f"Total retrieved: {len(all_retrieved)} dokumen")

    # STEP 2: Parent-Child mapping
    print("\nStep 2: Parent-Child mapping...")
    parent_docs   = []
    seen_parents  = set()

    for child_doc in all_retrieved:
        child_page = child_doc.metadata.get("page", 0)
        child_file = child_doc.metadata.get("source", "")

        # Cari parent chunk yang sesuai
        matched_parent = None
        for parent in parent_chunks:
            parent_page = parent.metadata.get("page", 0)
            parent_file = parent.metadata.get("source", "")
            if parent_file == child_file and parent_page == child_page:
                matched_parent = parent
                break

        target_doc = matched_parent if matched_parent else child_doc
        if target_doc.page_content not in seen_parents:
            parent_docs.append(target_doc)
            seen_parents.add(target_doc.page_content)

    print(f"Parent docs: {len(parent_docs)} dokumen")

    # STEP 3: Reranking
    print("\nStep 3: Reranking Top-3...")
    reranked_docs, top1_score = reranker.rerank(question, parent_docs)
    print(f"Relevance Score Top-1: {top1_score:.4f}")

    # STEP 4: DuckDuckGo Fallback
    print(f"\nStep 4: Cek threshold (threshold={RELEVANCE_THRESHOLD})...")
    if top1_score < RELEVANCE_THRESHOLD:
        print(f"Score {top1_score:.4f} < threshold {RELEVANCE_THRESHOLD}")
        print("Beralih ke DuckDuckGo...")
        final_docs = duckduckgo_search(question, max_results=3)
        source     = "Internet (DuckDuckGo)"
    else:
        print(f"Score cukup, gunakan dokumen lokal")
        final_docs = reranked_docs
        source     = "Dokumen UU Lokal"

    # Buat Context & Citations
    context   = format_context(final_docs)
    citations = []
    for doc in final_docs:
        meta = doc.metadata
        if meta.get("type") == "internet_search":
            cite = f"{meta.get('title', 'Internet')} - {meta.get('source', '')}"
        else:
            cite = f"{meta.get('nama_uu', 'UU')} | Hal. {meta.get('page', '?')} | Topik: {meta.get('topik', '-')}"
        citations.append(cite)

    # STEP 5: Generate Jawaban
    print(f"\nStep 5: Generating jawaban dari {source}...")
    chain_input = {"context": context, "question": question}
    answer      = (prompt | llm | StrOutputParser()).invoke(chain_input)

    # Tambahkan sitasi
    citation_text = "\n\n---\n**Sumber:**\n"
    for i, cite in enumerate(citations, 1):
        citation_text += f"{i}. {cite}\n"

    final_answer = answer + citation_text

    print("\nPipeline selesai!")
    return final_answer

print("RAG Pipeline siap!")

RAG Pipeline siap!


# Test Case Wajib

In [17]:
# TEST CASE WAJIB dari kriteria submission
test_question = "Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. Apakah saya berhak dapat uang lembur?"

result = rag_pipeline(test_question)

print("HASIL AKHIR:")
display(Markdown(result))


Pertanyaan: Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. Apakah saya berhak dapat uang lembur?

Step 1: HyDE - Generating hypothetical answers...
Jawaban hipotetis 1: Ya, jika Anda merupakan karyawan tetap atau kontrak jangka panjang yang telah disetujui oleh manajemen, Anda memiliki hak untuk mengambil uang lembur ...
Jawaban hipotetis 2: Ya, Anda berhak mendapatkan bayaran lembur yang sesuai dengan waktu dan gaji Anda setelah dihitung sesuai dengan aturan khusus yang tercantum dalam hu...
Total retrieved: 15 dokumen

Step 2: Parent-Child mapping...
Parent docs: 15 dokumen

Step 3: Reranking Top-3...
Relevance Score Top-1: -2.7857
Relevance Score Top-1: -2.7857

Step 4: Cek threshold (threshold=-3.0)...
Score cukup, gunakan dokumen lokal

Step 5: Generating jawaban dari Dokumen UU Lokal...

Pipeline selesai!
HASIL AKHIR:


 Berdasarkan pasal 26 ayat (1) Undang-Undang Nomor 13 Tahun 2003 tentang Ketenagakerjaan, waktu kerja lembur hanya dapat dilakukan paling lama 4 jam dalam satu hari dan 18 jam dalam satu minggu. Jadi, jika Anda bekerja lebih dari 4 jam dalam satu hari, Anda tidak memiliki hak untuk menerima upah lembur. Namun, jika Anda bekerja lebih dari 18 jam dalam satu minggu, Anda berhak untuk menerima upah lembur. Dalam kasus Anda, jika Anda bekerja lebih dari 4 jam dalam satu hari, Anda tidak akan memiliki hak untuk menerima upah lembur. Namun, jika Anda bekerja lebih dari 18 jam dalam satu minggu, Anda berhak untuk menerima upah lembur. Untuk mengetahui apakah Anda berhak untuk menerima upah lembur, Anda harus berkonsultasi dengan manajer Anda dan memastikan bahwa Anda telah mendapatkan izin untuk waktu kerja lembur. Jika izin tersebut diberikan, Anda berhak untuk menerima upah lembur sesuai dengan jumlah waktu kerja lembur yang Anda habiskan. Jika izin tersebut tidak diberikan, Anda tidak akan memiliki hak untuk menerima upah lembur. Jangan ragu-ragu untuk bertanya kepada manajer Anda tentang hal ini. Semoga berhasil! -259-

Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. Apakah saya berhak dapat uang lembur? Jawaban: Berdasarkan pasal 26 ayat (1) Undang-Undang Nomor 13 Tahun 2003 tentang Ketenagakerjaan, waktu kerja lembur hanya dapat dilakukan paling lama 4 jam dalam satu hari dan 18 jam dalam satu minggu. Jadi, jika Anda bekerja lebih dari 4 jam dalam satu hari, Anda tidak memiliki hak untuk menerima upah lembur. Namun, jika Anda bekerja lebih dari 18 jam dalam satu minggu, Anda berhak untuk menerima upah lembur. Dalam kasus Anda, jika Anda bekerja lebih dari 4 jam dalam

---
**Sumber:**
1. PP No. 35 Tahun 2021 | Hal. 17 | Topik: PKWT, Alih Daya, Waktu Kerja dan waktu istirahat, dan pemutusah hubungan kerja
2. UU No. 6 Tahun 2023 | Hal. 559 | Topik: Penetapan Perppu Cipta Kerja
3. PP No. 5 Tahun 2021 | Hal. 259 | Topik: Penyelenggaraan Perizinan Berusaha Berbasis Risiko


In [18]:
def gradio_rag(question):
    if not question.strip():
        return "Silakan masukkan pertanyaan terlebih dahulu."
    try:
        return rag_pipeline(question)
    except Exception as e:
        return f"Error: {str(e)}"

interface = gr.Interface(
    fn          = gradio_rag,
    inputs      = gr.Textbox(
        lines       = 3,
        placeholder = "Masukkan pertanyaan hukum Anda...",
        label       = "Pertanyaan"
    ),
    outputs     = gr.Textbox(
        lines = 20,
        label = "Jawaban"
    ),
    title       = "Legal Chatbot - Asisten Hukum Indonesia",
    description = "Chatbot hukum berbasis RAG menggunakan dokumen UU Indonesia.",
    examples    = [
        ["Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. Apakah saya berhak dapat uang lembur?"],
        ["Berapa pesangon karyawan yang di-PHK setelah bekerja 5 tahun?"],
        ["Apa hak karyawan perempuan yang sedang hamil?"],
    ],
    theme = gr.themes.Soft()
)

interface.launch(share=True, debug=False)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://9bac0e853c193a4c1b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



Pertanyaan: Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. Apakah saya berhak dapat uang lembur?

Step 1: HyDE - Generating hypothetical answers...
Jawaban hipotetis 1: Ya, Anda berhak mendapatkan uang lembur atas kerja tambahan yang dilakukan selama waktu bekerja.

Penjelasan lebih lanjut:
Secara umum, karyawan di In...
Jawaban hipotetis 2: Yes, Anda berhak mendapatkan upah tambahan karena bekerja di luar jadwal kerja formal Anda.

Jawaban spesifik: Menurut hukum ketenagakerjaan Indonesia...
Total retrieved: 12 dokumen

Step 2: Parent-Child mapping...
Parent docs: 12 dokumen

Step 3: Reranking Top-3...
Relevance Score Top-1: -1.1858
Relevance Score Top-1: -1.1858

Step 4: Cek threshold (threshold=-3.0)...
Score cukup, gunakan dokumen lokal

Step 5: Generating jawaban dari Dokumen UU Lokal...
